## How to run in Colab
1. Download the dataset zip [here](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%206%20-%20Applications%20for%20Data%20Analysis/W6D2%20-%20Advanced%20Numpy/globalpowerplantdatabasev130.zip) and save it in `C:\Users\sivan\Datasets`.
2. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
3. Run the first code cell and, when it asks, choose `globalpowerplantdatabasev130.zip` from `C:\Users\sivan\Datasets`.

**Run the cells from top to bottom** (Shift+Enter runs one cell).

In [ ]:
from google.colab import files
# colab tool for uploading files from my computer
uploaded = files.upload()
# opens a window where I choose the zip
!unzip -o "globalpowerplantdatabasev130.zip"
# unzips it, the csv I need is global_power_plant_database.csv

# Week 3 Day 2 – Daily Challenge: Global Power Plant Analysis

**Objective:** apply my knowledge of NumPy, Pandas and Matplotlib to analyze a real-world dataset, the Global Power Plant Database from the World Resources Institute.

**Tasks**
1. Data Import and Cleaning
2. Exploratory Data Analysis
3. Statistical Analysis
4. Time Series Analysis
5. Advanced Visualization
6. Matrix Operations in Real-World Context
7. Integrating NumPy with Pandas and Matplotlib

In [ ]:
import numpy as np
# numpy for math and arrays
import pandas as pd
# pandas for the table
import matplotlib.pyplot as plt
# matplotlib for charts
import seaborn as sns
# seaborn for nicer charts (heatmap, boxplot)
from scipy import stats
# scipy.stats for the hypothesis tests

## 1. Data Import and Cleaning
- Import the dataset using Pandas.
- Identify missing values and handle them appropriately.
- Use NumPy to convert relevant columns to numerical types if necessary.

In [ ]:
# low_memory=False stops a warning about mixed column types
df = pd.read_csv('global_power_plant_database.csv', low_memory=False)
print(df.shape)
df.head()

In [ ]:
# how many missing values in each column
df.isnull().sum()

**How I handle the missing values**
- `capacity_mw`, `latitude`, `longitude`, `primary_fuel` and `country` have no missing values, so I keep every plant.
- `other_fuel1/2/3`, `wepp_id`, `owner`, `url` and the `estimated_generation_note` columns I don't need, so I drop them.
- `commissioning_year` is missing for about half of the plants. I don't fill it with a made-up year (that would create a fake trend). I keep it as NaN and only use the plants that have a year in the time analysis.
- The `generation_gwh_*` columns are mostly empty, so I don't fill them either. I don't need them: I use `capacity_mw` as the power output.

In [ ]:
# I keep only the columns I use
cols = ['country', 'country_long', 'name', 'capacity_mw', 'latitude', 'longitude',
        'primary_fuel', 'commissioning_year', 'generation_gwh_2019']
plants = df[cols].copy()

# pd.to_numeric with errors='coerce' turns anything that isn't a number into NaN
plants['commissioning_year'] = pd.to_numeric(plants['commissioning_year'], errors='coerce')
# np.floor cuts the decimals (some years are like 1998.5), then I keep the column as float because of NaN
plants['commissioning_year'] = np.floor(plants['commissioning_year'])
# astype(np.float64) makes sure capacity and location are numbers
plants[['capacity_mw', 'latitude', 'longitude']] = plants[['capacity_mw', 'latitude', 'longitude']].astype(np.float64)

print(plants.dtypes)
print(plants.isnull().sum())

## 2. Exploratory Data Analysis
- Utilize Pandas to summarize key statistics (mean, median, standard deviation) for numerical columns.
- Explore the distribution of power plants by country and fuel type.

In [ ]:
# describe gives count, mean, std, min, quartiles and max
print(plants[['capacity_mw', 'commissioning_year', 'generation_gwh_2019']].describe().round(1))

# the median is not in describe by name (it is the 50% row), so I print it too
print('\nMedian:')
print(plants[['capacity_mw', 'commissioning_year', 'generation_gwh_2019']].median())

In [ ]:
# value_counts counts how many plants each fuel / country has
fuel_counts = plants['primary_fuel'].value_counts()
country_counts = plants['country_long'].value_counts().head(10)
print(fuel_counts)
print(country_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(fuel_counts.index, fuel_counts.values, color='steelblue')
axes[0].set_title('Number of Plants by Fuel Type')
axes[0].tick_params(axis='x', rotation=60)

axes[1].bar(country_counts.index, country_counts.values, color='orange')
axes[1].set_title('Top 10 Countries by Number of Plants')
axes[1].tick_params(axis='x', rotation=60)

plt.tight_layout()
plt.show()

## 3. Statistical Analysis
- Perform a statistical analysis of power output by fuel type using NumPy’s statistical functions.
- Use hypothesis testing to determine if the mean power output differs significantly between different fuel types.

*I use `capacity_mw` (the size of the plant in megawatts) as the power output, because it has no missing values.*

In [ ]:
# for each fuel I apply NumPy functions to the capacity of its plants
fuel_stats = plants.groupby('primary_fuel')['capacity_mw'].agg(['mean', 'median', 'std', 'sum', 'count'])
fuel_stats = fuel_stats.sort_values('mean', ascending=False).round(1)
fuel_stats

**Hypothesis test 1 – all fuels together (ANOVA, `f_oneway`)**
- H0: the mean capacity is the same for all fuel types.
- H1: at least one fuel type has a different mean capacity.

alpha = 0.05

In [ ]:
# one array of capacities for each fuel
groups = [plants[plants['primary_fuel'] == fuel]['capacity_mw'].values for fuel in fuel_counts.index]
f_stat, p_value = stats.f_oneway(*groups)
print('F-statistic:', round(f_stat, 2))
print('p-value:', p_value)
if p_value < 0.05:
    print('I reject H0: the mean capacity differs between fuel types.')
else:
    print('I fail to reject H0.')

**Hypothesis test 2 – Coal vs Solar (`ttest_ind`)**
- H0: Coal and Solar plants have the same mean capacity.
- H1: their mean capacities are different.

I use `equal_var=False` because the two groups have very different spreads.

In [ ]:
coal = plants[plants['primary_fuel'] == 'Coal']['capacity_mw']
solar = plants[plants['primary_fuel'] == 'Solar']['capacity_mw']
print('Coal mean:', round(np.mean(coal), 1), 'MW | Solar mean:', round(np.mean(solar), 1), 'MW')

t_stat, p_value = stats.ttest_ind(coal, solar, equal_var=False)
print('t-statistic:', round(t_stat, 2))
print('p-value:', p_value)
if p_value < 0.05:
    print('I reject H0: Coal and Solar plants have different mean capacity.')
else:
    print('I fail to reject H0.')

## 4. Time Series Analysis
- If the dataset includes time-related data (like year of establishment), use NumPy to analyze trends over time.
- Explore how the mix of fuel types for power generation has evolved over the years.

*I use `commissioning_year` (the year the plant started working) and only the plants that have it. I look at 1950 onwards, because before that there are very few plants.*

In [ ]:
# dropna removes plants without a year
years_df = plants.dropna(subset=['commissioning_year'])
years_df = years_df[years_df['commissioning_year'] >= 1950]
print('Plants with a year (1950+):', len(years_df))

# how many plants and how much capacity were added each year
per_year = years_df.groupby('commissioning_year').agg(plants=('name', 'count'), capacity=('capacity_mw', 'sum'))

# np.polyfit fits a straight line to the number of new plants per year
slope, intercept = np.polyfit(per_year.index, per_year['plants'], 1)
print('Trend: about', round(slope, 1), 'more new plants every year')

plt.figure(figsize=(12, 5))
plt.plot(per_year.index, per_year['plants'], label='New plants')
plt.plot(per_year.index, slope * per_year.index + intercept, linestyle='--', color='red', label='Linear trend (polyfit)')
plt.title('New Power Plants per Commissioning Year')
plt.xlabel('Year')
plt.ylabel('Number of plants')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# I group the years into decades: 1987 // 10 * 10 = 1980
years_df = years_df.copy()
years_df['decade'] = (years_df['commissioning_year'] // 10 * 10).astype(int)

# capacity added per decade for each fuel, I keep the 6 biggest fuels and put the rest in 'Other'
top_fuels = years_df.groupby('primary_fuel')['capacity_mw'].sum().sort_values(ascending=False).head(6).index
years_df['fuel_group'] = np.where(years_df['primary_fuel'].isin(top_fuels), years_df['primary_fuel'], 'Other')
mix = years_df.groupby(['decade', 'fuel_group'])['capacity_mw'].sum().unstack().fillna(0)

# divide each row by its total to get the share in percent
mix_pct = mix.div(mix.sum(axis=1), axis=0) * 100
print(mix_pct.round(1))

mix_pct.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Fuel Mix of New Capacity by Decade (%)')
plt.xlabel('Decade')
plt.ylabel('Share of new capacity (%)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

*Note: the "2020" bar is only the year 2020 (the data ends there), so it has few plants and is not a full decade.*

## 5. Advanced Visualization
- Create visualizations using Matplotlib and Seaborn to illustrate my findings.
- Consider plotting the geographical distribution of power plants using latitude and longitude data, if available.

In [ ]:
# each dot is one plant, x = longitude, y = latitude, so it looks like a world map
plt.figure(figsize=(14, 7))
for fuel in fuel_counts.index[:8]:
    part = plants[plants['primary_fuel'] == fuel]
    plt.scatter(part['longitude'], part['latitude'], s=3, alpha=0.5, label=fuel)
plt.title('Power Plants Around the World by Fuel Type (top 8 fuels)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(markerscale=5)
plt.show()

In [ ]:
# boxplot of capacity per fuel, I use a log scale because capacities go from 1 MW to over 20,000 MW
plt.figure(figsize=(12, 6))
sns.boxplot(data=plants, x='primary_fuel', y='capacity_mw', order=fuel_stats.index)
plt.yscale('log')
plt.title('Capacity by Fuel Type (log scale)')
plt.xlabel('Fuel')
plt.ylabel('Capacity (MW)')
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

## 6. Matrix Operations in Real-World Context
- Demonstrate matrix operations by analyzing relationships between different attributes (e.g., fuel type, capacity, and geographic location).
- Discuss the relevance of eigenvectors and eigenvalues in this context.

*Fuel type is text, so I turn it into a number column: 1 if the plant is renewable (Solar, Wind, Hydro, Biomass, Geothermal, Wave and Tidal), 0 if not.*

In [ ]:
renewables = ['Solar', 'Wind', 'Hydro', 'Biomass', 'Geothermal', 'Wave and Tidal']
# np.where gives 1 where the condition is True and 0 where it is False
plants['is_renewable'] = np.where(plants['primary_fuel'].isin(renewables), 1, 0)

# I take the 4 numeric attributes as a NumPy matrix (rows = plants, columns = attributes)
attrs = ['capacity_mw', 'latitude', 'longitude', 'is_renewable']
X = plants[attrs].to_numpy()
print('Matrix shape:', X.shape)

# np.corrcoef needs the attributes as rows, so I use the transpose X.T
corr = np.corrcoef(X.T)
corr_df = pd.DataFrame(corr, index=attrs, columns=attrs).round(3)
corr_df

In [ ]:
sns.heatmap(corr_df, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# eigenvalues and eigenvectors of the correlation matrix
eigenvalues, eigenvectors = np.linalg.eig(corr)
# the correlation matrix is symmetric, so the results are real numbers; .real removes the empty '+0j' part
eigenvalues = eigenvalues.real
eigenvectors = eigenvectors.real
print('Eigenvalues:', eigenvalues.round(3))
print('Eigenvectors (each column is one):')
print(pd.DataFrame(eigenvectors, index=attrs).round(3))

# share of the total variation explained by each eigenvalue
print('Share of variation (%):', (eigenvalues / eigenvalues.sum() * 100).round(1))

In [ ]:
# matrix multiplication example: the eigenvector times the matrix equals the eigenvalue times the eigenvector
v = eigenvectors[:, 0]
print('corr dot v:     ', np.dot(corr, v).round(3))
print('eigenvalue * v: ', (eigenvalues[0] * v).round(3))

**What the eigenvalues and eigenvectors mean here (in plain words)**
- The correlation matrix shows how the attributes move together. The eigenvectors are "directions" that combine the attributes, and each eigenvalue tells me how much of the variation in the data is in that direction.
- If all attributes were unrelated, every eigenvalue would be 1. Eigenvalues bigger than 1 mean some attributes are linked; eigenvalues smaller than 1 mean a direction carries little information.
- Here all the eigenvalues are fairly close to 1 (see the numbers above), which tells me capacity, location and renewable/non-renewable are only weakly related to each other. The biggest one mostly combines `capacity_mw` and `is_renewable` (renewable plants tend to be smaller).
- This is the idea behind PCA: keep the directions with the biggest eigenvalues to describe the data with fewer columns. With such weak correlations I can't shrink this data much without losing information.

## 7. Integrating NumPy with Pandas and Matplotlib
- Show how NumPy can be used to enhance data manipulation in Pandas and data visualization in Matplotlib.
- Provide examples, such as using NumPy arrays for complex filtering in Pandas or for creating sophisticated plots in Matplotlib.

In [ ]:
# complex filtering with NumPy boolean masks and &:
# big (>= 1000 MW) renewable plants in the southern hemisphere (latitude < 0)
cap = plants['capacity_mw'].to_numpy()
lat = plants['latitude'].to_numpy()
ren = plants['is_renewable'].to_numpy()
mask = (cap >= 1000) & (ren == 1) & (lat < 0)
print('Plants that match:', mask.sum())
plants[mask].sort_values('capacity_mw', ascending=False)[['name', 'country_long', 'primary_fuel', 'capacity_mw']].head(10)

In [ ]:
# np.where to make a size category column
plants['size'] = np.where(plants['capacity_mw'] >= 500, 'Large', np.where(plants['capacity_mw'] >= 50, 'Medium', 'Small'))
plants['size'].value_counts()

In [ ]:
# np.log10 makes the very skewed capacity readable in a histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(plants['capacity_mw'], bins=50, color='gray')
axes[0].set_title('Capacity (MW), raw')
axes[0].set_xlabel('MW')
axes[1].hist(np.log10(plants['capacity_mw']), bins=50, color='green')
axes[1].set_title('Capacity, log10 scale')
axes[1].set_xlabel('log10(MW)')
plt.tight_layout()
plt.show()
print('Skewness raw:', round(stats.skew(plants['capacity_mw']), 2), '| after log10:', round(stats.skew(np.log10(plants['capacity_mw'])), 2))

In [ ]:
# rolling mean (from the lesson) smooths the yearly number of new plants
smooth = per_year['plants'].rolling(window=5).mean()
plt.figure(figsize=(12, 5))
plt.plot(per_year.index, per_year['plants'], alpha=0.4, label='New plants per year')
plt.plot(per_year.index, smooth, color='red', label='5-year moving average')
plt.title('New Plants per Year with 5-Year Moving Average')
plt.xlabel('Year')
plt.ylabel('Number of plants')
plt.legend()
plt.grid(True)
plt.show()

## Findings
*All numbers are from my run of this notebook (the data is real, so they don't change).*

- **The data:** 34,936 plants in 167 countries. The USA has the most plants (9,833), then China (4,235) and the UK (2,751). About half of the plants have no commissioning year.
- **Most plants are small:** the median capacity is only 16.7 MW but the mean is 163.4 MW, because a few huge plants pull the mean up (skewness 8.9; after log10 it is only 0.51).
- **Count vs size:** Solar has the most plants (10,665) but they are tiny (mean 17.7 MW). Nuclear (mean 2,092 MW) and Coal (mean 844 MW) have the biggest plants, and Coal has the largest total capacity (about 1.97 million MW).
- **Hypothesis tests:** ANOVA across all fuels (F = 967, p ≈ 0) and Coal vs Solar t-test (t = 44.9, p ≈ 0) both reject H0: mean capacity really differs by fuel.
- **Trend over time:** the number of new plants per year goes up by about 9 plants a year since 1950. Hydro was about 47% of new capacity in the 1950s and only 12% in the 2010s, while Wind went from 0% to about 9% and Coal stayed the biggest (about 50%) until 2019.
- **Correlations are weak:** the strongest is capacity vs renewable (-0.32: renewable plants are smaller). All eigenvalues are between 0.61 and 1.37, so the attributes are almost independent.
- **Map:** the scatter of longitude/latitude draws the world, with most plants in the USA, Europe and East Asia. The 42 large (1,000 MW+) renewable plants in the southern hemisphere are almost all Hydro, mostly in Brazil (Tucuruí 8,535 MW, Itaipu 7,000 MW).

## How I used NumPy, Pandas and Matplotlib
- **Pandas:** loading the CSV, finding missing values with `isnull`, `dropna`, `describe`, `value_counts`, `groupby`/`agg` per fuel and per year, and the correlation table.
- **NumPy:** `mean`/`median`/`std`, `np.where` and boolean masks with `&` for filtering and new columns, `np.floor` and `astype` for numbers, `np.polyfit` for the trend line, `np.corrcoef` and `np.linalg.eig` for the matrix part, `np.log10` for the histogram.
- **Matplotlib / Seaborn:** bar charts, line charts with trend and moving average, a stacked bar of the fuel mix, a latitude/longitude scatter map, a boxplot, a heatmap and histograms.
- **SciPy:** `f_oneway` and `ttest_ind` for the hypothesis tests, `skew` for the log check.